# Multimodal Cancer Modeling in the Age of Foundation Model Embeddings
### Steven Song\*, Morgan Borjigin-Wang\*, Irene R. Madejski, Robert L. Grossman

\* Equal contribution

Read our paper here: https://proceedings.mlr.press/v297/song26a.html

***

The Cancer Genome Atlas (TCGA) has enabled novel discoveries and served as a large-scale reference dataset in cancer through its harmonized genomics, clinical, and imaging data. Numerous prior studies have developed bespoke deep learning models over TCGA for tasks such as cancer survival prediction. A modern paradigm in biomedical deep learning is the development of foundation models (FMs) to derive feature embeddings agnostic to a specific modeling task. Biomedical text especially has seen growing development of FMs. While TCGA contains free-text data as pathology reports, these have been historically underutilized.

* **We investigate the ability to train classical machine learning models over multimodal, zero-shot FM embeddings of cancer data.**
* We demonstrate the ease and additive effect of multimodal fusion, outperforming unimodal models.
* Overall, we propose a simple, modernized approach to multimodal cancer modeling using FM embeddings.

### Overview

<img src="https://raw.githubusercontent.com/StevenSong/multimodal-cancer-modeling/refs/heads/main/overview.png" alt="conceptual overview figure" width="50%"/>

Conceptually, the proposed framework does late fusion of unimodal models trained over their respective embeddings. Specifically, we use:
* BulkRNABert ([Gélard et al. 2025)](https://proceedings.mlr.press/v259/gelard25a.html)) for RNA-seq data
* UNI2-h ([Chen et al. 2024](https://www.nature.com/articles/s41591-024-02857-3)) for histology data
* BioMistral ([Labrak et al. 2024](https://aclanthology.org/2024.findings-acl.348/)) for pathology report data (summarized by Llama-3.1-8B-Instruct ([Grattafiori et al. 2024](https://arxiv.org/abs/2407.21783)))

#### Use the GDC API to get TCGA case metadata

In [1]:
import requests
import numpy as np
import pandas as pd
from io import StringIO

In [2]:
# these filters were generated using our GDC Cohort Copilot with these queries:
# - "samples from TCGA"
# - "TCGA samples with diagnostic slides as svs files"
# - "TCGA samples with RNA-sequences gene expression quantification data as tsv files"
# - "TCGA samples with pathology report pdf files"
# give it a try: https://m3aicommons.org/Analysis
tcga_filter = {"op": "and", "content": [{"op": "in", "content": {"field": "cases.project.program.name", "value": ["TCGA"]}}]}
hist_filter = {"op": "and", "content": [{"op": "in", "content": {"field": "cases.project.program.name", "value": ["TCGA"]}}, {"op": "in", "content": {"field": "files.experimental_strategy", "value": ["Diagnostic Slide"]}}, {"op": "in", "content": {"field": "files.data_format", "value": ["svs"]}}]}
expr_filter = {"op": "and", "content": [{"op": "in", "content": {"field": "cases.project.program.name", "value": ["TCGA"]}}, {"op": "in", "content": {"field": "files.experimental_strategy", "value": ["RNA-Seq"]}}, {"op": "in", "content": {"field": "files.data_type", "value": ["Gene Expression Quantification"]}}, {"op": "in", "content": {"field": "files.data_format", "value": ["tsv"]}}]}
text_filter = {"op": "and", "content": [{"op": "in", "content": {"field": "cases.project.program.name", "value": ["TCGA"]}}, {"op": "in", "content": {"field": "files.data_type", "value": ["Pathology Report"]}}, {"op": "in", "content": {"field": "files.data_format", "value": ["pdf"]}}]}

In [3]:
# case metadata
response = requests.post(
    "https://api.gdc.cancer.gov/cases",
    json={
        "filters": tcga_filter,
        "fields": ",".join(["project.project_id", "submitter_id", "diagnoses.age_at_diagnosis", "diagnoses.diagnosis_is_primary_disease", "demographic.sex_at_birth", "demographic.race", "demographic.ethnicity"]),
        "format": "JSON",
        "size": str(100_000),
    },
)

hits = []
for hit in response.json()["data"]["hits"]:
    proj = hit.pop("project", {})
    demo = hit.pop("demographic", {})
    dxs = hit.pop("diagnoses", [])
    for dx in dxs:
        if dx.get("diagnosis_is_primary_disease", False) and dx["age_at_diagnosis"] is not None and not np.isnan(dx["age_at_diagnosis"]):
            hit["age_at_diagnosis"] = dx["age_at_diagnosis"]
            break
    if "age_at_diagnosis" in hit:
        hits.append(hit | proj | demo)
metadata = pd.DataFrame(hits).drop(columns=["id"])
assert metadata["age_at_diagnosis"].notna().all()
metadata["age_in_years"] = metadata["age_at_diagnosis"] / 365.24 # convert age to years
metadata["age_binned"] = pd.cut(metadata["age_in_years"], bins=[0, 20, 40, 60, 80, 100]) # convert age to 20-year bins
metadata

,submitter_id,age_at_diagnosis,project_id,race,ethnicity,sex_at_birth,age_in_years,age_binned
0,TCGA-E9-A5FL,24053,TCGA-BRCA,white,not hispanic or latino,female,65.855328,"(60, 80]"
1,TCGA-A2-A1G4,25966,TCGA-BRCA,white,not hispanic or latino,female,71.092980,"(60, 80]"
2,TCGA-BH-A0HF,28233,TCGA-BRCA,white,not reported,female,77.299858,"(60, 80]"
3,TCGA-AR-A1AS,20028,TCGA-BRCA,asian,not reported,female,54.835177,"(40, 60]"
4,TCGA-D8-A13Y,19028,TCGA-BRCA,white,not hispanic or latino,female,52.097251,"(40, 60]"
...,...,...,...,...,...,...,...,...
11066,TCGA-A5-A0GQ,28078,TCGA-UCEC,white,not hispanic or latino,female,76.875479,"(60, 80]"
11067,TCGA-FI-A2F4,23621,TCGA-UCEC,black or african american,not hispanic or latino,female,64.672544,"(60, 80]"
11068,TCGA-EO-A3KU,25116,TCGA-UCEC,not reported,not reported,female,68.765743,"(60, 80]"
11069,TCGA-A5-A0GM,19452,TCGA-UCEC,white,not hispanic or latino,female,53.258132,"(40, 60]"


In [4]:
# survival data
response = requests.post(
    "https://api.gdc.cancer.gov/analysis/survival",
    json={"filters": tcga_filter},
)
rows = response.json()["results"][0]["donors"]
survival = pd.DataFrame(rows).drop(columns=["id", "project_id"])
survival

,time,censored,survivalEstimate,submitter_id
0,1.0,False,1.000000,TCGA-AA-3492
1,1.0,True,1.000000,TCGA-C8-A275
2,1.0,True,1.000000,TCGA-AC-A7VC
3,1.0,True,1.000000,TCGA-FV-A495
4,1.0,False,1.000000,TCGA-CG-4306
...,...,...,...,...
11079,9634.0,True,0.163994,TCGA-WB-A80P
11080,10346.0,False,0.163994,TCGA-EE-A2GD
11081,10870.0,False,0.122996,TCGA-FS-A1ZC
11082,11217.0,True,0.081997,TCGA-LH-A9QB


In [5]:
# get mapping of file ID --> case ID
def get_file_mapping(cohort_filter):
    response = requests.post(
        "https://api.gdc.cancer.gov/files",
        json={
            "filters": cohort_filter,
            "fields": ",".join(["file_name", "cases.project.project_id", "cases.submitter_id"]),
            "format": "TSV",
            "size": str(100_000),
        },
    )
    df = pd.read_csv(StringIO(response.text), sep="\t")
    df = df.rename(columns={"cases.0.project.project_id": "project_id", "cases.0.submitter_id": "submitter_id"}) # these files all map to a single case
    df["file_name"] = df["file_name"].str.replace(".svs", "").str.replace(".rna_seq.augmented_star_gene_counts.tsv", "").str.replace(".PDF", "")
    assert df["file_name"].is_unique
    return df.set_index("submitter_id")["file_name"]

hist_mapping = get_file_mapping(hist_filter)
expr_mapping = get_file_mapping(expr_filter)
text_mapping = get_file_mapping(text_filter)

#### Load embeddings from the Gen3 Embedding Service

In [6]:
# TODO replace this entire cell with getting the embeddings from the API
# !wget "https://uchicago.box.com/shared/static/k8z0kip2pej2v62pwgymdm45gt7dq0yw.h5" -O "data/hist.h5"
# !wget "https://uchicago.box.com/shared/static/hr82b5c9g3h4y8c7avrnbgvhgdcoetld.h5" -O "data/expr.h5"
# !wget "https://uchicago.box.com/shared/static/liwt3vlvdpmbfsa21wqboshh9nv6enm2.h5" -O "data/summ.h5"

import h5py
from tqdm import tqdm

expr_file = "data/expr.h5" # BulkRNABert
hist_file = "data/hist.h5" # UNI2
text_file = "data/summ.h5" # BioMistral - Summarized

expr_embs = dict()
hist_embs = dict()
text_embs = dict()

for fpath, embs, do_upper in tqdm([
    (expr_file, expr_embs, False),
    (hist_file, hist_embs, False),
    (text_file, text_embs, True),
]):
    with h5py.File(fpath, "r") as h5:
        for case_id in h5.keys():
            for sample_fname in h5[case_id].keys():
                emb_key = sample_fname
                if do_upper:
                    emb_key = emb_key.upper()
                embs[emb_key] = h5[case_id][sample_fname][:]

  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:03<00:00,  1.04s/it]


#### Align metadata and embeddings

In [7]:
case_ids = (
    # case IDs that have all 3 embedding modalities
    set(expr_mapping[expr_mapping.isin(expr_embs)].index) &
    set(hist_mapping[hist_mapping.isin(hist_embs)].index) &
    set(text_mapping[text_mapping.isin(text_embs)].index)
)

assert metadata["submitter_id"].is_unique and survival["submitter_id"].is_unique
df = metadata.merge(survival, on="submitter_id")
df = df[df["submitter_id"].isin(case_ids)]
df = df.sort_values(["project_id", "submitter_id"])
df = df.set_index("submitter_id")
df

,age_at_diagnosis,project_id,race,ethnicity,sex_at_birth,age_in_years,age_binned,time,censored,survivalEstimate
submitter_id,,,,,,,,,,
TCGA-OR-A5J1,21496,TCGA-ACC,white,not reported,male,58.854452,"(40, 60]",1355.0,False,0.645345
TCGA-OR-A5J2,16090,TCGA-ACC,white,hispanic or latino,female,44.053225,"(40, 60]",1677.0,False,0.590849
TCGA-OR-A5J3,8624,TCGA-ACC,white,hispanic or latino,female,23.611872,"(20, 40]",2091.0,True,0.536054
TCGA-OR-A5J5,11171,TCGA-ACC,white,hispanic or latino,male,30.585369,"(20, 40]",365.0,False,0.882847
TCGA-OR-A5J6,10839,TCGA-ACC,black or african american,hispanic or latino,female,29.676377,"(20, 40]",2703.0,True,0.473733
...,...,...,...,...,...,...,...,...,...,...
TCGA-YZ-A980,27716,TCGA-UVM,white,not hispanic or latino,male,75.884350,"(60, 80]",1862.0,True,0.564878
TCGA-YZ-A982,28938,TCGA-UVM,white,not hispanic or latino,female,79.230095,"(60, 80]",495.0,True,0.833581
TCGA-YZ-A983,18769,TCGA-UVM,white,not hispanic or latino,female,51.388128,"(40, 60]",798.0,True,0.750574


In [8]:
# align ordering of embeddings to case order in metadata
expr_X = []
hist_X = []
text_X = []
for case_id in tqdm(df.index):
    for embs, mapping, X in [
        (expr_embs, expr_mapping, expr_X),
        (hist_embs, hist_mapping, hist_X),
        (text_embs, text_mapping, text_X),
    ]:
        fnames = mapping.loc[case_id]
        if isinstance(fnames, str): # single file, guaranteed to be in embs
            emb = embs[fnames]
        else: # multiple files, at least one will be in embs
            emb = np.mean([embs[f] for f in fnames if f in embs], axis=0)
        X.append(emb)
expr_X = np.asarray(expr_X)
hist_X = np.asarray(hist_X)
text_X = np.asarray(text_X)

100%|██████████| 7801/7801 [00:03<00:00, 2510.02it/s]


#### Preprocess data before modeling

In [9]:
from typing import Optional
from sklearn.decomposition import PCA
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

In [10]:
demo_X = OneHotEncoder(drop="if_binary", sparse_output=False, dtype=np.float32).fit_transform(df[["sex_at_birth", "age_binned", "race", "ethnicity"]])
canc_X = OneHotEncoder(drop="if_binary", sparse_output=False, dtype=np.float32).fit_transform(df[["project_id"]])
y = np.asarray(
    list(zip(~df["censored"], df["time"])),
    dtype=[("Status", "?"), ("Survival_in_days", "<f8")],
)

In [11]:
# stratify by observed mortality and cancer type
splitter = df["censored"].astype(str) + "_" + df["project_id"]

# split all data modalities into train/test
(
    demo_X_train, demo_X_test,
    canc_X_train, canc_X_test,
    expr_X_train, expr_X_test,
    hist_X_train, hist_X_test,
    text_X_train, text_X_test,
    y_train,      y_test,
) = train_test_split(
    demo_X, canc_X, expr_X, hist_X, text_X, y,
    test_size=0.2,
    random_state=42,
    stratify=splitter,
)

In [12]:
# this one helper will be used to run both unimodal and multimodal experiments
def train_eval_model(
    *,  # enforce kwargs
    X_train: np.ndarray, y_train: np.ndarray,
    X_test: np.ndarray, y_test: np.ndarray,
    pca_components: Optional[int], standardize: bool,
) -> dict:
    print("Training survival model")

    # z-score input features
    if standardize:
        print("--standardized")
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

    # dimensionality reduction
    if pca_components is not None:
        print("--reduced")
        pca = PCA(n_components=pca_components, random_state=42)
        X_train = pca.fit_transform(X_train)
        X_test = pca.transform(X_test)

    # fit survival model
    cox = CoxPHSurvivalAnalysis(alpha=0.1).fit(X_train, y_train)
    print("--trained")

    # generate predictions
    y_train_pred = cox.predict(X_train)
    y_test_pred = cox.predict(X_test)

    # evaluate predictions
    c_index = concordance_index_censored(
        event_indicator=y_test["Status"],
        event_time=y_test["Survival_in_days"],
        estimate=y_test_pred,
    )[0]

    return {
        "c_index": c_index,
        "y_test_pred": y_test_pred,
        "y_train_pred": y_train_pred,
    }

#### Train unimodal models

In [13]:
demo_results = train_eval_model(X_train=demo_X_train, y_train=y_train, X_test=demo_X_test, y_test=y_test, pca_components=None, standardize=False)
canc_results = train_eval_model(X_train=canc_X_train, y_train=y_train, X_test=canc_X_test, y_test=y_test, pca_components=None, standardize=False)
expr_results = train_eval_model(X_train=expr_X_train, y_train=y_train, X_test=expr_X_test, y_test=y_test, pca_components=256, standardize=True)
hist_results = train_eval_model(X_train=hist_X_train, y_train=y_train, X_test=hist_X_test, y_test=y_test, pca_components=256, standardize=True)
text_results = train_eval_model(X_train=text_X_train, y_train=y_train, X_test=text_X_test, y_test=y_test, pca_components=256, standardize=True)

Training survival model
--trained
Training survival model
--trained
Training survival model
--standardized
--reduced
--trained
Training survival model
--standardized
--reduced
--trained
Training survival model
--standardized
--reduced
--trained


#### Train multimodal model
The unimodal models' predicted risk scores are used as input to the multimodal fusion model.

In [14]:
fuse_X_train = np.asarray([
    demo_results["y_train_pred"],
    canc_results["y_train_pred"],
    expr_results["y_train_pred"],
    hist_results["y_train_pred"],
    text_results["y_train_pred"],
]).T

fuse_X_test = np.asarray([
    demo_results["y_test_pred"],
    canc_results["y_test_pred"],
    expr_results["y_test_pred"],
    hist_results["y_test_pred"],
    text_results["y_test_pred"],
]).T

fuse_results = train_eval_model(X_train=fuse_X_train, y_train=y_train, X_test=fuse_X_test, y_test=y_test, pca_components=None, standardize=True)

Training survival model
--standardized
--trained


#### Evaluation
Our multimodal fusion approach substantially beats all unimodal results!

In [15]:
results = pd.Series({
    "demo": demo_results["c_index"],
    "canc": canc_results["c_index"],
    "expr": expr_results["c_index"],
    "hist": hist_results["c_index"],
    "text": text_results["c_index"],
    "fuse": fuse_results["c_index"],
}, name="C-index")

print("Unimodal Results")
print("------------------")
display(results.loc[["demo", "canc", "expr", "hist", "text"]])

print("Multimodal Results")
print("------------------")
display(results.loc[["fuse"]])

Unimodal Results
------------------


demo    0.620357
canc    0.746400
expr    0.752201
hist    0.759262
text    0.744995
Name: C-index, dtype: float64

Multimodal Results
------------------


fuse    0.791706
Name: C-index, dtype: float64